In [ ]:
!unzip -q /content/drive/MyDrive/HAM10000 -d /content/

In [ ]:
# Install libraries for google colab.
!pip install segmentation-models-pytorch torch torchmetrics albumentations

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# PyTorch libraries
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Segmentation Models PyTorch
import segmentation_models_pytorch as smp

# Albumentations for data augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Torchmetrics for evaluation metrics
from torchmetrics.classification import BinaryJaccardIndex, BinaryAccuracy

In [ ]:
# Set up the device to use GPU if avaliable, otherwise cpu.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, images_dir, masks_dir, image_names, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        # Get image and mask paths
        image_name = self.image_names[idx]
        image_path = os.path.join(self.images_dir, image_name)
        mask_name = image_name.replace('.jpg', '_segmentation.png')
        mask_path = os.path.join(self.masks_dir, mask_name)

        # Load image and mask
        image = np.array(Image.open(image_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'), dtype=np.float32)
        # Ensure mask is binary (0 or 1)
        mask[mask > 0] = 1.0

        # Apply transformations
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        else:
            image = TF.to_tensor(image)
            mask = torch.tensor(mask).unsqueeze(0)

        return image, mask

In [ ]:
# Define transformations for images and masks.
train_transforms = A.Compose([
    A.Resize(576, 448),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(576, 448),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# Paths to the images and masks directory.
images_dir = '/content/HAM10000/images/'
masks_dir = '/content/HAM10000/masks/'

# Get list of image names. Ignore files that are not .jpg or .png (as macOS adds hidden files).
image_names = [f for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]

# Split into training and validation sets
train_names, val_names = train_test_split(image_names, test_size=0.2, random_state=42)

# Create dataset instances
train_dataset = SkinLesionDataset(images_dir, masks_dir, train_names, transform=train_transforms)
val_dataset = SkinLesionDataset(images_dir, masks_dir, val_names, transform=val_transforms)

In [ ]:
# Create data loaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init

class Attention_block(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        """
        F_g : number of channels in gating signal (from decoder)
        F_l : number of channels in the encoder (skip connection)
        F_int: intermediate number of channels (typically F_l // 2)
        """
        super(Attention_block, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# =============================================================================
# Custom U-Net with ResNet34 Encoder & Attention Gates on Skip Connections
# =============================================================================
import torchvision.models as models

class UNetResNet34_Attn(nn.Module):
    def __init__(self, n_classes):
        super(UNetResNet34_Attn, self).__init__()
        resnet = models.resnet34(pretrained=True)
        # Encoder
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)    # [B, 64, H/2, W/2]
        self.layer1 = nn.Sequential(resnet.maxpool, resnet.layer1)             # [B, 64, H/4, W/4]
        self.layer2 = resnet.layer2                                           # [B, 128, H/8, W/8]
        self.layer3 = resnet.layer3                                           # [B, 256, H/16, W/16]
        self.layer4 = resnet.layer4                                           # [B, 512, H/32, W/32]

        # Attention Gates for skip connections
        self.attn3 = Attention_block(F_g=256, F_l=256, F_int=128)   # For skip x3
        self.attn2 = Attention_block(F_g=128, F_l=128, F_int=64)    # For skip x2
        self.attn1 = Attention_block(F_g=64, F_l=64, F_int=32)      # For skip x1
        self.attn0 = Attention_block(F_g=64, F_l=64, F_int=32)      # For skip x0 (optional)

        # Decoder: Upsampling and convolution blocks
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)  # Upsample bottleneck
        self.decoder3 = nn.Sequential(
            nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = nn.Sequential(
            nn.Conv2d(128 + 128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.upconv1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.decoder0 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder forward pass
        x0 = self.layer0(x)     # [B, 64, H/2, W/2]
        x1 = self.layer1(x0)    # [B, 64, H/4, W/4]
        x2 = self.layer2(x1)    # [B, 128, H/8, W/8]
        x3 = self.layer3(x2)    # [B, 256, H/16, W/16]
        x4 = self.layer4(x3)    # [B, 512, H/32, W/32]

        # Decoder with Attention Gates on Skip Connections
        d4 = self.upconv4(x4)   # [B, 256, H/16, W/16]
        # Apply attention gate: gating signal d4 and skip connection x3
        x3_att = self.attn3(g=d4, x=x3)
        d4 = torch.cat([d4, x3_att], dim=1)  # [B, 256+256, H/16, W/16]
        d4 = self.decoder3(d4)              # [B, 256, H/16, W/16]

        d3 = self.upconv3(d4)   # [B, 128, H/8, W/8]
        x2_att = self.attn2(g=d3, x=x2)
        d3 = torch.cat([d3, x2_att], dim=1)  # [B, 128+128, H/8, W/8]
        d3 = self.decoder2(d3)              # [B, 128, H/8, W/8]

        d2 = self.upconv2(d3)   # [B, 64, H/4, W/4]
        x1_att = self.attn1(g=d2, x=x1)
        d2 = torch.cat([d2, x1_att], dim=1)  # [B, 64+64, H/4, W/4]
        d2 = self.decoder1(d2)              # [B, 64, H/4, W/4]

        d1 = self.upconv1(d2)   # [B, 64, H/2, W/2]
        x0_att = self.attn0(g=d1, x=x0)
        d1 = torch.cat([d1, x0_att], dim=1)  # [B, 64+64, H/2, W/2]
        d1 = self.decoder0(d1)              # [B, 64, H/2, W/2]

        out = self.final_conv(d1)           # [B, n_classes, H/2, W/2]
        # Upsample to match original image resolution (576x448)
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=True)
        return out

In [ ]:
model = UNetResNet34_Attn(n_classes=1)
model = model.to(device)

loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

iou_metric = BinaryJaccardIndex().to(device)
accuracy_metric = BinaryAccuracy().to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 158MB/s]


In [ ]:
num_epochs = 25
score_history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'val_iou': [],
    'val_accuracy': []
}

In [ ]:
patience = 5
best_val_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    avg_train_loss = train_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    total_iou = 0.0
    total_acc = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device).float().unsqueeze(1)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item() * images.size(0)
            outputs = torch.sigmoid(outputs)
            preds = (outputs > 0.5).float()
            preds_flat = preds.view(-1)
            masks_flat = masks.view(-1)
            iou = iou_metric(preds_flat, masks_flat)
            acc = accuracy_metric(preds_flat, masks_flat)
            total_iou += iou.item() * images.size(0)
            total_acc += acc.item() * images.size(0)
    avg_val_loss = val_loss / len(val_loader.dataset)
    avg_iou = total_iou / len(val_loader.dataset)
    avg_acc = total_acc / len(val_loader.dataset)

    # Save scores for this epoch
    score_history['epoch'].append(epoch + 1)
    score_history['train_loss'].append(avg_train_loss)
    score_history['val_loss'].append(avg_val_loss)
    score_history['val_iou'].append(avg_iou)
    score_history['val_accuracy'].append(avg_acc)

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}, Val IoU: {avg_iou:.4f}, Val Accuracy: {avg_acc:.4f}")

Epoch [1/25]
Train Loss: 0.1390
Val Loss: 0.0775, Val IoU: 0.8573, Val Accuracy: 0.9602
Epoch [2/25]
Train Loss: 0.0691
Val Loss: 0.0694, Val IoU: 0.8713, Val Accuracy: 0.9638
Epoch [3/25]
Train Loss: 0.0598
Val Loss: 0.0648, Val IoU: 0.8795, Val Accuracy: 0.9666
Epoch [4/25]
Train Loss: 0.0552
Val Loss: 0.0647, Val IoU: 0.8795, Val Accuracy: 0.9661
Epoch [5/25]
Train Loss: 0.0498
Val Loss: 0.0613, Val IoU: 0.8856, Val Accuracy: 0.9674
Epoch [6/25]
Train Loss: 0.0462
Val Loss: 0.0599, Val IoU: 0.8879, Val Accuracy: 0.9690
Epoch [7/25]
Train Loss: 0.0429
Val Loss: 0.0606, Val IoU: 0.8867, Val Accuracy: 0.9684
Epoch [8/25]
Train Loss: 0.0394
Val Loss: 0.0594, Val IoU: 0.8889, Val Accuracy: 0.9685
Epoch [9/25]
Train Loss: 0.0378
Val Loss: 0.0591, Val IoU: 0.8892, Val Accuracy: 0.9687
Epoch [10/25]
Train Loss: 0.0355
Val Loss: 0.0574, Val IoU: 0.8923, Val Accuracy: 0.9699
Epoch [11/25]
Train Loss: 0.0329
Val Loss: 0.0559, Val IoU: 0.8949, Val Accuracy: 0.9705
Epoch [12/25]
Train Loss: 0.03

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
df_scores = pd.DataFrame(score_history)
df_scores.to_csv("training_scores.csv", index=False)
print("Training scores saved.")

Training scores saved.
